# PaliGemma-3B H2 cross-ablation (D78)

Does ablating PaliGemma's P circuit defeat the typographic override, and does
ablating its T circuit defeat the prompted prior? **D73's design, unchanged**,
on PaliGemma (DECISIONS.md D78, written before this notebook): the same head
sets and layer-matched random draws (`cross_ablation.build_plan`), the same
tests and wording, analysed by the same `scripts/analyze_cross_ablation.py`.

**One change of method, exact: layer replay** (`circuit_overlap.replay`). An
ablation at the last position of layer l cannot change anything entering
layers 0..l, so per object one unablated pass records what enters each decoder
layer, and each ablated configuration recomputes only the layers from its own
earliest ablated layer. Section 4 checks it against full forward passes with
D73's own ablation hook on every configuration of one P object and one T
object, and checks the batching for row leaks, before any loop. Every object
in the loops also replays its unablated run and is flagged if it does not
reproduce the full pass.

**Runtime ~12-15 h estimated (D78); section 4 measures it.** Phases: zero
ablation on P (121 objects) then T (302), mean ablation, clean image. Each
session stops starting new objects at `SESSION_BUDGET_H` and saves what is
done; to continue, attach the previous version's output (its `ckpt/` folder)
and run again.

**Untested on the GPU.** Settings: GPU T4, Internet on, `HF_TOKEN` ticked.
Run as **Save Version -> Save & Run All**.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/SohamRShinde/VLM-Overlap-RP"
REPO_DIR = "/kaggle/working/research_paper"

import importlib, os, subprocess, sys

if REPO_URL:
    if os.path.exists(REPO_DIR):
        # Always update an existing checkout: a stale clone silently runs old
        # code, and the failure shows up later as a confusing ImportError.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "-q", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# Drop any already-imported package modules: Python caches them, so after an
# update the kernel would keep serving the old file.
for _m in [m for m in list(sys.modules) if m.startswith("circuit_overlap")]:
    del sys.modules[_m]
importlib.invalidate_caches()

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("repo at", COMMIT)   # record this in RESULTS.md with any number from this run

# Fail here, loudly, if the checkout is older than this notebook expects.
import importlib.util as _ilu
_need = {}
for _mod, _names in {
    "circuit_overlap.paligemma": ["CHECKPOINT", "load_paligemma", "build_inputs", "text_geometry",
                                  "FullForwardRunner", "square_image_block",
                                  "last_token_reaches_earlier_positions", "quiet_processor_warning"],
    "circuit_overlap.replay": ["LayerReplay", "ablate_rows_last_token", "replay_groups",
                               "start_layer", "ReplayMismatch"],
    "circuit_overlap.cross_ablation": ["build_plan", "ablate_heads_last_token",
                                       "analyse_cross_ablation", "source_flips"],
    "circuit_overlap.patching": ["capture_last_token_o_proj_inputs"],
    "circuit_overlap.replication": ["lietzow_rows", "color_token_sets", "run_cached"],
    "circuit_overlap.t_patching": ["render_t_pair", "LAYOUT", "load_patching_records"],
    "circuit_overlap.t_screen": ["PRIOR_PROMPT", "VISUAL_PROMPT", "color_first_token_ids",
                                 "load_notes_config", "native_bf16"],
    "circuit_overlap.heads": ["load_head_set", "format_head_label"],
    "circuit_overlap.provenance": ["result_envelope", "write_result"],
}.items():
    if _ilu.find_spec(_mod) is None:
        _need[_mod] = ["module"]
        continue
    _m = importlib.import_module(_mod)
    _gone = [n for n in _names if not hasattr(_m, n)]
    if _gone:
        _need[_mod] = _gone
for _f in ("configs/heads_lietzow_paligemma2_3b.json",
           "results/pg_replication_patching_paligemma2_3b.json",
           "results/pg_t_patching_paligemma2_3b.json",
           "results/pg_t_patching_records_paligemma2_3b.jsonl.gz",
           "results/pg_replication_knockout_paligemma2_3b.json",
           "scripts/analyze_pg_cross_ablation.py"):
    if not os.path.exists(f"{REPO_DIR}/{_f}"):
        _need[_f] = ["file"]
import inspect as _insp
if not _need:   # D76a additions the name check cannot see
    from circuit_overlap.replication import run_cached as _rc
    from circuit_overlap.paligemma import FullForwardRunner as _ffr
    for _what, _ok in (("run_cached(budget_s=...)", "budget_s" in _insp.signature(_rc).parameters),
                       ("FullForwardRunner(last_only=...)", "last_only" in _insp.signature(_ffr).parameters)):
        if not _ok:
            _need[_what] = ["parameter"]
if _need:
    raise RuntimeError(f"checkout {COMMIT} is missing {_need}. "
                       "Push the local commits to origin/main, then re-run this cell.")
print("code is current for this notebook")

In [ ]:
# Use one GPU. On "GPU T4 x2", device_map="auto" splits the model across both
# cards, which only adds transfer time for a 3B model. This must run before
# torch is first imported, so restart the kernel if torch is already loaded.
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# bitsandbytes is required for the 4-bit load (D10). Installing it into a
# kernel that already imported transformers does nothing: restart and re-run.
!pip install -q -U "transformers>=4.49" accelerate datasets inflect "bitsandbytes>=0.46.1"
import torch, transformers
from transformers.utils import is_bitsandbytes_available
from circuit_overlap.t_screen import native_bf16
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
# Not torch.cuda.is_bf16_supported(): it says True on a T4 (emulation).
print("bitsandbytes available:", is_bitsandbytes_available(),
      "| native bf16:", native_bf16(torch.cuda.get_device_capability(0)))

from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    # Secrets are attached PER NOTEBOOK: one on your account is not enough.
    raise RuntimeError(
        "HF_TOKEN is not attached to this notebook. In the editor: Add-ons -> "
        "Secrets -> tick the checkbox next to HF_TOKEN (exact label), then "
        "restart the session and run from cell 1. It must be ticked before "
        "Save & Run All, or the background run fails here."
    ) from exc

## 2. Model, geometry, and Lietzow's heads

As `pg_replication.ipynb`.

In [ ]:
import json
from circuit_overlap.paligemma import CHECKPOINT, load_paligemma, quiet_processor_warning, text_geometry
from circuit_overlap.heads import load_head_set, format_head_label
from circuit_overlap.t_screen import PRIOR_PROMPT, VISUAL_PROMPT

quiet_processor_warning()   # the per-call `<image>` nag; section 3 checks the input itself

model, processor = load_paligemma()
COMPUTE_DTYPE = str(model.config.quantization_config.bnb_4bit_compute_dtype).replace("torch.", "")
print(CHECKPOINT, "| compute dtype:", COMPUTE_DTYPE, "(D76 (3): fp16 on a T4)")
print("devices used:", sorted({str(d) for d in getattr(model, "hf_device_map", {"": model.device}).values()}))

geo = text_geometry(model)          # raises if o_proj's width != n_heads x head_dim
n_layers, n_heads, head_dim = geo["n_layers"], geo["n_heads"], geo["head_dim"]
print(geo)

HEADS_FILE = f"{REPO_DIR}/configs/heads_lietzow_paligemma2_3b.json"
hs = load_head_set(HEADS_FILE, n_layers=n_layers, n_heads_per_layer=n_heads)   # raises if out of range
hs_raw = json.load(open(HEADS_FILE))
print("their promoting :", [format_head_label(h) for h in hs.promoting])
print("their suppressing:", [format_head_label(h) for h in hs.suppressing])
print("they report:", hs_raw["group_ablation_flip_pct"], "on", hs_raw["n_correctly_conflicting"],
      "objects; single heads", hs.single_head_prior_flip_pct)

## 3. [VERIFY] the checks that already passed in `pg_replication` / `pg_t_patching`

Prompt and image block, `labels` dropped without changing the logits, hooks in
a full forward pass, the T renders, last-position-only logits, and the timing
of a full forward pass.

In [ ]:
import glob, inflect, pathlib, shutil, time
import torch
from collections import Counter
from IPython.display import display
from datasets import load_dataset
from circuit_overlap.paligemma import (
    FullForwardRunner, build_inputs as _pg_inputs, square_image_block,
    last_token_reaches_earlier_positions,
)
from circuit_overlap.patching import capture_last_token_o_proj_inputs
from circuit_overlap.cross_ablation import ablate_heads_last_token
from circuit_overlap.replication import lietzow_rows
from circuit_overlap.t_screen import color_first_token_ids, load_notes_config
from circuit_overlap.t_patching import LAYOUT, render_t_pair

WORK = pathlib.Path("/kaggle/working/pg_cross_ablation")
CKPT = WORK / "ckpt"            # checkpoints and saved activations: NOT results
(CKPT / "acts").mkdir(parents=True, exist_ok=True)
# Resume: bring back ckpt/ (zero_*.jsonl, mean_*.jsonl, clean.jsonl, acts/) from
# an earlier run attached as an input. Found automatically; set RESUME_FROM to
# the folder holding those files only if the search picks the wrong one.
RESUME_FROM = None          # e.g. "/kaggle/input/pg-cross-ablation-ckpt"
if RESUME_FROM is None:
    hits = sorted(glob.glob("/kaggle/input/**/zero_P.jsonl", recursive=True))
    RESUME_FROM = str(pathlib.Path(hits[0]).parent) if hits else None
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, CKPT, dirs_exist_ok=True)
for name in ("zero_P.jsonl", "zero_T.jsonl", "mean_P.jsonl", "mean_T.jsonl", "clean.jsonl"):
    n_lines = sum(1 for l in open(CKPT / name) if l.strip()) if (CKPT / name).exists() else 0
    print(f"{name}: {n_lines} objects already done")
print(f"acts/: {len(list((CKPT / 'acts').glob('*.pt')))} files" + (f" (from {RESUME_FROM})" if RESUME_FROM else ""))

ds = load_dataset("mgolov/Visual-Counterfact", split="color")
_eng = inflect.engine()
rows, dropped = lietzow_rows(ds, _eng)
print(len(rows), "objects", dict(dropped), "(the Qwen runs used the same 469)")
vocab = sorted({c for r in rows for c in r["correct"]} | {r["attack"] for r in rows})
color_ids, collisions = color_first_token_ids(processor.tokenizer, vocab)
ambiguous = set(collisions)
print(len(vocab), "colours | sharing a first token on the Gemma tokenizer (D26):", collisions)

def build_inputs(image, prompt):
    return _pg_inputs(processor, image, prompt, model.device)

def prompts(r):
    return (VISUAL_PROMPT.format(object=r["name"]),
            PRIOR_PROMPT.format(a_object=_eng.a(r["name"])))

notes_cfg = load_notes_config(f"{REPO_DIR}/configs/notes.json")
spec = next(s for s in notes_cfg["specs"] if s.name == LAYOUT)
neutral_words = notes_cfg["neutral_words"]

def t_inputs(r):
    """(incongruent inputs, neutral inputs, render info), as t_patching.ipynb."""
    row = ds[r["i"]]
    pr = render_t_pair(row["original_image"], row["counterfact_image"], r["attack"], neutral_words, spec)
    prompt = VISUAL_PROMPT.format(object=r["name"])
    return build_inputs(pr["incongruent"], prompt), build_inputs(pr["neutral"], prompt), pr

VERIFY = {"compute_dtype": COMPUTE_DTYPE, "geometry": geo, "ambiguous_colours": sorted(ambiguous)}
IMAGE_TOKEN_ID = processor.image_token_id      # their get_image_token_id

# [VERIFY] 1-2. the prompt and the image block.
r0 = rows[0]
inp0 = build_inputs(ds[r0["i"]]["counterfact_image"], prompts(r0)[1])
ids0 = inp0["input_ids"][0].tolist()
offset0, side0 = square_image_block(ids0, IMAGE_TOKEN_ID)
text0 = processor.tokenizer.decode(ids0[offset0 + side0 * side0:])
VERIFY["image_block"] = {"offset": offset0, "side": side0, "n_tokens": len(ids0),
                         "pixel_values": list(inp0["pixel_values"].shape)}
VERIFY["prompt_after_image"] = text0
print("inputs:", {k: tuple(v.shape) for k, v in inp0.items()})
print("image block:", VERIFY["image_block"])
print("text after the image:", repr(text0))

# [VERIFY] 7. labels: the first run passed them; this run drops them.
inp0_labels = _pg_inputs(processor, ds[r0["i"]]["counterfact_image"], prompts(r0)[1], model.device,
                         keep_labels=True)
VERIFY["processor_returns_labels"] = "labels" in inp0_labels
if VERIFY["processor_returns_labels"]:
    with torch.no_grad():
        with_labels = model(**inp0_labels, use_cache=False).logits[0, -1].float()
        without = model(**inp0, use_cache=False).logits[0, -1].float()
    VERIFY["labels_max_dlogit"] = float((with_labels - without).abs().max())
    print("logits with vs without labels, max |dlogit|:", VERIFY["labels_max_dlogit"], "(expect ~0)")
    assert VERIFY["labels_max_dlogit"] < 0.05, (
        "labels change the logits (attention mask): the first run's checkpoints are not comparable -- "
        "stop and report this")
    del with_labels, without
del inp0_labels
assert ids0[-1] != IMAGE_TOKEN_ID, "the last token must be text"
assert prompts(r0)[1] in text0, "the prompt is not what the model reads"

# [VERIFY] 3. colour tokens: colliding colours are excluded (D26), and reported.
print("excluded colours (D26):", sorted(ambiguous) or "none")

# [VERIFY] 4. hooks in a full forward pass.
runner0 = FullForwardRunner(model, inp0)
base0 = runner0.logits()
with ablate_heads_last_token(model, [], head_dim):
    same0 = runner0.logits()
with ablate_heads_last_token(model, hs.promoting, head_dim):
    knocked0 = runner0.logits()
with capture_last_token_o_proj_inputs(model) as acts0:
    runner0.logits()
VERIFY["empty_ablation_max_dlogit"] = float((same0 - base0).abs().max())
VERIFY["promoting_ablation_max_dlogit"] = float((knocked0 - base0).abs().max())
print("empty ablation max |dlogit|:", VERIFY["empty_ablation_max_dlogit"], "(expect ~0)")
print("promoting ablation max |dlogit|:", VERIFY["promoting_ablation_max_dlogit"], "(expect > 0)")
assert VERIFY["empty_ablation_max_dlogit"] < 1e-3
assert VERIFY["promoting_ablation_max_dlogit"] > 1e-3
assert sorted(acts0) == list(range(n_layers)) and all(
    acts0[l].shape[-1] == n_heads * head_dim for l in acts0), "captured head outputs have the wrong shape"
del acts0

# [VERIFY] 5. the T renders.
s0_, n0_, pr0 = t_inputs(r0)
print(f"{r0['name']}: note {r0['attack']!r} vs neutral {pr0['neutral_word']!r}")
display(pr0["neutral"].resize((256, 256))); display(pr0["incongruent"].resize((256, 256)))
assert pr0["same_note_footprint"], "the two notes differ in shape: not a single-factor contrast"
assert s0_["input_ids"].shape == n0_["input_ids"].shape, "renders tokenise to different lengths"

# Attention direction (reported, D76 (2)).
VERIFY["last_token_vs_position_0"] = last_token_reaches_earlier_positions(model, inp0, position=0)
VERIFY["last_token_vs_first_text"] = last_token_reaches_earlier_positions(
    model, inp0, position=offset0 + side0 * side0)
for k in ("last_token_vs_position_0", "last_token_vs_first_text"):
    print(k, VERIFY[k])

# [VERIFY] 6. last-position-only logits (runner default) == the full output's last row.
full0 = FullForwardRunner(model, inp0, last_only=False).logits()
VERIFY["logits_to_keep_max_dlogit"] = float((full0 - base0).abs().max())
print("last-only vs full logits, max |dlogit|:", VERIFY["logits_to_keep_max_dlogit"], "(expect ~0)")
assert VERIFY["logits_to_keep_max_dlogit"] < 0.05

# Speed, and the total, before any long loop (CLAUDE.md).
from circuit_overlap.paligemma import _repeat_rows

def _time(fn, reps=2):
    fn(); torch.cuda.synchronize(); t0 = time.time()
    for _ in range(reps):
        fn()
    torch.cuda.synchronize()
    return (time.time() - t0) / reps

step = _time(runner0.logits)
print(f"one forward pass (last-only logits): {step:.2f}s")
VERIFY["forward_s"] = step
per_row = {1: step}
for b in (4, 8, 16):
    try:
        batch_in = _repeat_rows(inp0, b)
        with torch.no_grad():
            per_row[b] = _time(lambda: model(**batch_in, use_cache=False, logits_to_keep=1)) / b
        print(f"batch {b:2d}: {per_row[b]:.2f}s per row")
    except torch.cuda.OutOfMemoryError:
        print(f"batch {b:2d}: out of memory")
        break
    finally:
        batch_in = None
        torch.cuda.empty_cache()
BATCH = min(per_row, key=per_row.get)
VERIFY["per_row_s"], VERIFY["batch_size"] = per_row, BATCH
del runner0, base0, same0, knocked0, full0

## 4. The plan, the objects, and [VERIFY] the layer replay (D78 (3))

The plan is D73's, built from PaliGemma's committed discovery files. Then the
replay is checked; **the notebook stops if any check fails**:

1. on one P object and one T object, **every zero-ablation configuration** of
   the plan, on both runs where the plan asks for both: replay vs a full
   forward pass with `cross_ablation.ablate_heads_last_token` (D73's hook,
   independent code). Same argmax, same D67 flip outcome, max |dlogit| < 0.5;
2. mean ablation (non-zero values) on H_P: replay vs full forward pass;
3. no leaks between rows: an unablated row inside an ablated batch equals the
   unablated logits; each row of a batch equals the same configuration
   replayed alone; a reversed batch gives the same per-configuration logits;
4. the clean-image run, unablated and with H_P, replay vs full pass.

Then the replay is timed and the total estimated.

In [ ]:
import gzip, json, math
from circuit_overlap.cross_ablation import build_plan, ablate_heads_last_token
from circuit_overlap.replay import LayerReplay, ReplayMismatch, replay_groups, start_layer
from circuit_overlap.replication import color_token_sets
from circuit_overlap.t_patching import load_patching_records

P_FILE = f"{REPO_DIR}/results/pg_replication_patching_paligemma2_3b.json"
T_FILE = f"{REPO_DIR}/results/pg_t_patching_paligemma2_3b.json"
T_RECORDS = f"{REPO_DIR}/results/pg_t_patching_records_paligemma2_3b.jsonl.gz"
p_res, t_res = json.load(open(P_FILE))["result"], json.load(open(T_FILE))["result"]
plan = build_plan(p_res, t_res, n_heads_per_layer=n_heads)
configs = {k: [tuple(h) for h in v] for k, v in plan["configs"].items()}
for k, v in plan["sets"].items():
    print(f"  {k:7s} ({len(v)}): {', '.join(v)}")
print(len(plan["excluded_from_draws"]), "classified heads excluded from the random draws (D78: 15)")
FIRST = min(s for s in (start_layer(h) for h in configs.values()) if s is not None)
print("earliest ablated layer in the plan:", FIRST)

row_by_i = {r["i"]: r for r in rows}
p_disc = {r["i"]: (r["next_token_idx_prior"], r["next_token_idx_visual"])
          for r in p_res["records"] if r["status"] == "patched"}
t_disc = {r["i"]: (r["next_token_idx_prior"], r["next_token_idx_visual"])
          for r in load_patching_records(T_RECORDS)}
todo = {"P": [row_by_i[i] for i in p_disc], "T": [row_by_i[i] for i in t_disc]}
disc = {"P": p_disc, "T": t_disc}
clean_ids = sorted(set(p_disc) | set(t_disc))
print(f"P {len(todo['P'])} objects, T {len(todo['T'])}, clean {len(clean_ids)} (D78: 121, 302, 354)")
assert (len(todo["P"]), len(todo["T"]), len(clean_ids)) == (121, 302, 354)

def inputs(cond, r):
    """(source inputs, target inputs) for one object, as in its discovery run."""
    row = ds[r["i"]]
    if cond == "P":
        image = row["counterfact_image"]
        visual_prompt, prior_prompt = prompts(r)
        return build_inputs(image, prior_prompt), build_inputs(image, visual_prompt)
    src_in, tgt_in, _ = t_inputs(r)
    return src_in, tgt_in

def pair(logits, a):
    """[logit of the source run's answer, logit of the target run's answer] (D67)."""
    return [round(float(logits[a[0]]), 4), round(float(logits[a[1]]), 4)]

def source_wins(p):
    return not p[1] > p[0]        # ties go to the source, as D67 / knockout_flip_counts

def finite(d):
    return all(math.isfinite(x) for v in d.values() if isinstance(v, list) for x in v)

def replay_entries(rep, names, values=None, batch=None):
    """{config name: last-position logits} for names, batched by start layer."""
    out = {}
    for start, chunk in replay_groups(names, configs, batch or BATCH, FIRST):
        logits = rep.replay([configs[n] for n in chunk], start, head_dim, values)
        for n, row in zip(chunk, logits):
            out[n] = row
    return out

def full_ablated(inp, heads, values=None):
    with ablate_heads_last_token(model, heads, head_dim, values):
        return FullForwardRunner(model, inp).logits()

VERIFY["replay"] = {}

# [VERIFY] 1. every zero configuration, replay vs full forward, one object per condition.
for cond in ("P", "T"):
    r = todo[cond][0]
    src_in, tgt_in = inputs(cond, r)
    reps = {"prior": LayerReplay(model, src_in, FIRST), "visual": LayerReplay(model, tgt_in, FIRST)}
    a = (int(reps["prior"].logits.argmax()), int(reps["visual"].logits.argmax()))
    assert a == tuple(disc[cond][r["i"]]), f"{cond} {r['name']}: answers differ from discovery"
    worst, n_cmp, n_flip_mismatch, n_argmax_mismatch = 0.0, 0, 0, 0
    for kind, inp in (("prior", src_in), ("visual", tgt_in)):
        names = [n for n, runs in plan["zero"][cond] if kind == "prior" or runs == "both"]
        fast = replay_entries(reps[kind], names)
        for n in names:
            slow = full_ablated(inp, configs[n])
            worst = max(worst, float((slow - fast[n]).abs().max()))
            n_argmax_mismatch += int(slow.argmax()) != int(fast[n].argmax())
            n_flip_mismatch += source_wins(pair(slow, a)) != source_wins(pair(fast[n], a))
            n_cmp += 1
    VERIFY["replay"][f"all_zero_configs_{cond}"] = {
        "object": r["name"], "n_runs": n_cmp, "max_dlogit": worst,
        "argmax_mismatches": n_argmax_mismatch, "flip_mismatches": n_flip_mismatch,
        "softcap": [rp_.softcap for rp_ in reps.values()],
        "head_check": [rp_.head_check for rp_ in reps.values()]}
    print(cond, VERIFY["replay"][f"all_zero_configs_{cond}"])
    assert n_flip_mismatch == 0 and n_argmax_mismatch == 0 and worst < 0.5, "layer replay != full forward"

# The head's final soft-capping, decided on the first objects; every later replay uses it,
# and every object's own unablated replay (the per-object check) confirms it.
caps = {c for v in VERIFY["replay"].values() for c in v["softcap"]}
assert len(caps) == 1, f"soft-capping differs between runs: {caps}"
SOFTCAP = caps.pop()
VERIFY["replay"]["softcap"] = SOFTCAP
print("final logit soft-capping applied by the model's head:", SOFTCAP)

# [VERIFY] 2. mean-style values (non-zero): H_P set to the target run's own activations.
r = todo["P"][0]
src_in, tgt_in = inputs("P", r)
with capture_last_token_o_proj_inputs(model) as acts_t:
    FullForwardRunner(model, tgt_in).logits()
vals = {(l, h): acts_t[l][h * head_dim:(h + 1) * head_dim] for l, h in configs["P"]}
rep = LayerReplay(model, src_in, FIRST, softcap=SOFTCAP)
fast = rep.replay([configs["P"]], start_layer(configs["P"]), head_dim, vals)[0]
slow = full_ablated(src_in, configs["P"], vals)
VERIFY["replay"]["values_path"] = float((slow - fast).abs().max())
print("values path, max |dlogit|:", VERIFY["replay"]["values_path"])
assert VERIFY["replay"]["values_path"] < 0.5 and int(slow.argmax()) == int(fast.argmax())

# [VERIFY] 3. rows do not leak into each other.
s0 = start_layer(configs["P"])
names3 = ["none", "P", "rP_000", "rP_001"]
batch = rep.replay([configs[n] for n in names3], s0, head_dim)
alone = {n: rep.replay([configs[n]], s0, head_dim)[0] for n in names3}
rev = rep.replay([configs[n] for n in names3[::-1]], s0, head_dim).flip(0)
VERIFY["replay"]["unablated_row_in_ablated_batch"] = float((batch[0] - rep.logits).abs().max())
VERIFY["replay"]["batch_vs_alone"] = max(float((batch[k] - alone[n]).abs().max()) for k, n in enumerate(names3))
VERIFY["replay"]["batch_vs_reversed"] = float((batch - rev).abs().max())
VERIFY["replay"]["ablated_rows_differ"] = float((batch[1] - batch[0]).abs().max())
print({k: VERIFY["replay"][k] for k in ("unablated_row_in_ablated_batch", "batch_vs_alone",
                                        "batch_vs_reversed", "ablated_rows_differ")})
assert VERIFY["replay"]["unablated_row_in_ablated_batch"] < 0.5
assert VERIFY["replay"]["batch_vs_alone"] < 0.5 and VERIFY["replay"]["batch_vs_reversed"] < 0.5
assert VERIFY["replay"]["ablated_rows_differ"] > 0.1, "ablation had no effect in the batch"

# [VERIFY] 4. the clean image.
r = row_by_i[clean_ids[0]]
clean_in = build_inputs(ds[r["i"]]["original_image"], VISUAL_PROMPT.format(object=r["name"]))
rep = LayerReplay(model, clean_in, FIRST, softcap=SOFTCAP)
fast = replay_entries(rep, ["none", "P"])
VERIFY["replay"]["clean"] = max(float((fast["none"] - FullForwardRunner(model, clean_in).logits()).abs().max()),
                                float((fast["P"] - full_ablated(clean_in, configs["P"])).abs().max()))
print("clean image, max |dlogit|:", VERIFY["replay"]["clean"])
assert VERIFY["replay"]["clean"] < 0.5
del rep, reps, acts_t, vals

# Batch size and the total (D78 runtime): replay from FIRST at a few batch sizes.
rep = LayerReplay(model, src_in, FIRST, softcap=SOFTCAP)
rp = [configs["P"]]
per_row = {}
for b in (4, 8, 16):
    try:
        per_row[b] = _time(lambda: rep.replay(rp * b, FIRST, head_dim)) / b
        print(f"replay batch {b:2d} from layer {FIRST}: {per_row[b]:.3f}s per row")
    except torch.cuda.OutOfMemoryError:
        print(f"replay batch {b:2d}: out of memory")
        torch.cuda.empty_cache()
        break
BATCH = min(per_row, key=per_row.get)
per_layer_row = per_row[BATCH] / (n_layers - FIRST)
record_s = _time(lambda: LayerReplay(model, src_in, FIRST, softcap=SOFTCAP))
def layers_of(name):
    s = start_layer(configs[name])
    return n_layers - (FIRST if s is None else s)
n_obj = {"P": len(todo["P"]), "T": len(todo["T"])}
layer_rows = sum(n_obj[c] * (2 if runs == "both" else 1) * layers_of(n)
                 for ph in ("zero", "mean") for c in n_obj for n, runs in plan[ph][c])
layer_rows += len(clean_ids) * sum(layers_of(n) for n in plan["clean"])
n_records = 2 * 2 * sum(n_obj.values()) + len(clean_ids)
sentinel = (2 * sum(n_obj.values())) * (n_layers - FIRST)        # each run's unablated replay
est_h = ((layer_rows + sentinel) * per_layer_row + n_records * record_s) / 3600
VERIFY["replay"].update(batch=BATCH, per_row_s=per_row, record_s=record_s, estimate_h=est_h)
print(f"batch {BATCH}; recording pass {record_s:.2f}s; estimate ~{est_h:.1f} h in total")
del rep

## 5. Phase 1: zero ablation (P, then T)

Per object: the two unablated passes (recorded; their answers must equal
discovery's, or the object is dropped as `drift`), their last-token
activations saved for phase 2's means, a replay of each unablated run from
the earliest layer (flagged `replay_mismatch` if it does not reproduce the
full pass), then every zero-ablation configuration of the plan.

In [ ]:
import time
from circuit_overlap.replication import run_cached
SESSION_BUDGET_H = 11.0   # stop starting new objects after this many hours (Kaggle stops at 12 h)
DEADLINE = time.time() + SESSION_BUDGET_H * 3600
REPLAY_TOL = 0.1          # the per-object unablated-replay check (|dlogit|)

def left():
    return DEADLINE - time.time()

def run_configs(reps, entries, a, key_prefix="", values=None):
    """Each configuration's ablated logit pair, on the source run and, if asked, the target."""
    out = {}
    for kind in ("prior", "visual"):
        names = [n for n, runs in entries if kind == "prior" or runs == "both"]
        for n, logits in replay_entries(reps[kind], names, values).items():
            out[f"{key_prefix}{n}/{kind}"] = pair(logits, a)
    return out

def replay_check(reps):
    """The unablated run replayed from FIRST vs its own full pass (max |dlogit|)."""
    return max(float((rep.replay([[]], FIRST, head_dim)[0] - rep.logits).abs().max())
               for rep in reps.values())

def zero_object(cond):
    def fn(r):
        out = {"condition": cond, "name": r["name"]}
        src_in, tgt_in = inputs(cond, r)
        if src_in["input_ids"].shape != tgt_in["input_ids"].shape:
            return {"status": "shape_mismatch", **out}
        acts = {}
        reps = {}
        for kind, inp in (("prior", src_in), ("visual", tgt_in)):
            with capture_last_token_o_proj_inputs(model) as acts[kind]:
                reps[kind] = LayerReplay(model, inp, FIRST, softcap=SOFTCAP)
        a = (int(reps["prior"].logits.argmax()), int(reps["visual"].logits.argmax()))
        out.update(a_prior=a[0], a_visual=a[1])
        if a != tuple(disc[cond][r["i"]]):
            return {"status": "drift", **out, "discovery": list(disc[cond][r["i"]])}
        out["replay_check"] = round(replay_check(reps), 4)
        if out["replay_check"] > REPLAY_TOL:
            return {"status": "replay_mismatch", **out}
        torch.save({"source": torch.stack([acts["prior"][l] for l in range(n_layers)]).half().cpu(),
                    "target": torch.stack([acts["visual"][l] for l in range(n_layers)]).half().cpu()},
                   CKPT / "acts" / f"{cond}_{r['i']}.pt")
        out.update(run_configs(reps, plan["zero"][cond], a))
        out["status"] = "ok" if finite(out) else "numerics_failed"      # plan risk 7
        return out
    return fn

zero = {}
for cond in ("P", "T"):
    zero[cond] = run_cached(CKPT / f"zero_{cond}.jsonl", todo[cond], zero_object(cond),
                            progress_every=5, budget_s=max(left(), 0))
    print(cond, f"{len(zero[cond])}/{len(todo[cond])} done", dict(Counter(r["status"] for r in zero[cond])))
PHASE1_DONE = all(len(zero[c]) == len(todo[c]) for c in ("P", "T"))

## 6. Phase 2: mean ablation (D73 (10))

Each ablated head's slice set to its mean last-token activation over both runs
of all of that condition's objects (from phase 1's saved activations); the two
primary tests only. Runs only once phase 1 is complete.

In [ ]:
def condition_means(cond):
    total, n = None, 0
    for r in zero[cond]:
        if r["status"] != "ok":
            continue
        path = CKPT / "acts" / f"{cond}_{r['i']}.pt"
        if not path.exists():
            raise FileNotFoundError(f"{path} is missing: attach the previous run's whole ckpt/ folder")
        d = torch.load(path)
        for run in ("source", "target"):
            x = d[run].float()
            total = x if total is None else total + x
            n += 1
    return total / n                     # (n_layers, n_heads * head_dim)

def mean_object(cond, values):
    a_of = {r["i"]: (r["a_prior"], r["a_visual"]) for r in zero[cond] if r["status"] == "ok"}
    def fn(r):
        if r["i"] not in a_of:
            return {"status": "not_ok_in_phase_1", "condition": cond}
        src_in, tgt_in = inputs(cond, r)
        reps = {"prior": LayerReplay(model, src_in, FIRST, softcap=SOFTCAP),
                "visual": LayerReplay(model, tgt_in, FIRST, softcap=SOFTCAP)}
        out = {"condition": cond, **run_configs(reps, plan["mean"][cond], a_of[r["i"]],
                                                key_prefix="mean:", values=values)}
        out["status"] = "ok" if finite(out) else "numerics_failed"
        return out
    return fn

mean = {"P": [], "T": []}
if PHASE1_DONE:
    for cond in ("P", "T"):
        m = condition_means(cond).to(model.device)
        needed = {h for name, _ in plan["mean"][cond] for h in configs[name]}
        values = {(l, h): m[l, h * head_dim:(h + 1) * head_dim] for l, h in needed}
        mean[cond] = run_cached(CKPT / f"mean_{cond}.jsonl", todo[cond], mean_object(cond, values),
                                progress_every=10, budget_s=max(left(), 0))
        print(cond, f"{len(mean[cond])}/{len(todo[cond])} done", dict(Counter(r["status"] for r in mean[cond])))
else:
    print("phase 1 not complete: phase 2 not run this session")
PHASE2_DONE = PHASE1_DONE and all(len(mean[c]) == len(todo[c]) for c in ("P", "T"))

## 7. Phase 3: clean image (D73 (9))

The original image, the Visual prompt, no note: is the top-1 still a real
colour with each set (and the first 20 draws of each primary null) ablated?
Every object in P or T; objects whose colours share a first token are skipped.

In [ ]:
def clean_object(r):
    out = {"condition": "clean", "name": r["name"]}
    if ambiguous & ({r["attack"]} | set(r["correct"])):
        return {"status": "ambiguous", **out}
    image = ds[r["i"]]["original_image"]
    rep = LayerReplay(model, build_inputs(image, VISUAL_PROMPT.format(object=r["name"])), FIRST,
                      softcap=SOFTCAP)
    out["replay_check"] = round(float((rep.replay([[]], FIRST, head_dim)[0] - rep.logits).abs().max()), 4)
    if out["replay_check"] > REPLAY_TOL:
        return {"status": "replay_mismatch", **out}
    _, real = color_token_sets(r, color_ids)
    for name, logits in replay_entries(rep, plan["clean"]).items():
        if not torch.isfinite(logits).all():
            return {"status": "numerics_failed", **out}
        out[f"{name}/correct"] = int(logits.argmax()) in real
    out["status"] = "ok"
    return out

clean = run_cached(CKPT / "clean.jsonl", [row_by_i[i] for i in clean_ids], clean_object,
                   progress_every=25, budget_s=max(left(), 0))
PHASE3_DONE = len(clean) == len(clean_ids)
print(f"{len(clean)}/{len(clean_ids)} done", dict(Counter(r["status"] for r in clean)),
      f"| correct unablated: {sum(r.get('none/correct', False) for r in clean)}")

## 8. Save (after every session)

Merges the phases (an object counts only if every phase is `ok` for it, as
D73's notebook) and writes the summary and records. A **preliminary** D73
analysis is printed only once every phase is complete; the reported numbers
come from `scripts/analyze_cross_ablation.py` on the two saved files.

In [ ]:
from circuit_overlap.cross_ablation import analyse_cross_ablation
from circuit_overlap.provenance import result_envelope, write_result

merged = {}
for cond in ("P", "T"):
    by_i = {r["i"]: r for r in mean[cond]}
    merged[cond] = []
    for r in zero[cond]:
        rec = {**r, "condition": cond}
        m = by_i.get(r["i"], {})
        rec.update({k: v for k, v in m.items() if k.startswith("mean:")})
        if rec["status"] == "ok" and m.get("status") != "ok":
            rec["status"] = f"phase2_{m.get('status', 'missing')}"
        merged[cond].append(rec)

complete = PHASE1_DONE and PHASE2_DONE and PHASE3_DONE
if complete:
    prelim = analyse_cross_ablation(plan, merged, clean, n_boot=200)
    for name, t in prelim["primary"].items():
        print(f"PRELIMINARY {name}: {t['count']}/{t['n']} = {t['observed']:.1%} vs random "
              f"{t['null_mean']:.1%} (max {t['null_max']:.1%}), p = {t['p']:.4f} -> {t['wording']}")
else:
    print(f"phases complete: zero {PHASE1_DONE}, mean {PHASE2_DONE}, clean {PHASE3_DONE} -- "
          "no preliminary analysis; attach this version's output and run again")

summary = {
    "plan": plan,
    "complete": complete,
    "status": {c: dict(Counter(r["status"] for r in merged[c])) for c in merged},
    "clean_status": dict(Counter(r["status"] for r in clean)),
    "replay_check_max": max([r.get("replay_check", 0.0) for c in merged for r in merged[c]]
                            + [r.get("replay_check", 0.0) for r in clean] + [0.0]),
    "verify": VERIFY,
    "records_file": "pg_cross_ablation_records_paligemma2_3b.jsonl.gz",
}
print(write_result(WORK / "pg_cross_ablation_paligemma2_3b.json", result_envelope(
    "pg_cross_ablation", summary, model="PaliGemma2-3B-mix-448", checkpoint=CHECKPOINT, seed=0,
    prompt=f"P: prior={PRIOR_PROMPT!r} vs visual={VISUAL_PROMPT!r}; T: visual, incongruent vs neutral note",
    n=sum(len(v) for v in merged.values()), repo_dir=REPO_DIR, decision="D78",
    quantisation="4-bit nf4, double quant", compute_dtype=COMPUTE_DTYPE,
    gpu=torch.cuda.get_device_name(0), image_tokens=side0 * side0, layout=LAYOUT,
    ablation="o_proj input slice at last token, zero (phases 1, 3) or condition mean (phase 2); "
             "exact layer replay from the earliest ablated layer, checked against full forward passes")))
with gzip.open(WORK / "pg_cross_ablation_records_paligemma2_3b.jsonl.gz", "wt") as f:
    for cond in ("P", "T"):
        for r in merged[cond]:
            f.write(json.dumps(r) + "\n")
    for r in clean:
        f.write(json.dumps(r) + "\n")
print("\nDownload these two into results/ (ckpt/ is for resuming, not results):")
for name in ("pg_cross_ablation_paligemma2_3b.json", "pg_cross_ablation_records_paligemma2_3b.jsonl.gz"):
    print(f"  {name:48s} {(WORK / name).stat().st_size / 1e6:6.2f} MB")